# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate all record sets in the dataset and then explore their fields (columns). This helps us know the available structures and columns to reference by their `@id`s for further processing.

In [ ]:
print("Available record sets and fields (by @id):\n")
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets found in schema. Please check dataset structure.")
else:
    for record_set in record_sets:
        print(f"Record Set: {record_set['@id']}, name: {record_set.get('name', '')}")
        # List fields (columns) for this record set
        fields = record_set.get('field', [])
        if isinstance(fields, dict):  # Only one field
            fields = [fields]
        for field in fields:
            # For referencing, always use @id
            if isinstance(field, dict):
                print(f"    Field: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
            else:
                # Sometimes only @id is given
                print(f"    Field: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We will programmatically extract *all* available record sets from the dataset, using their `@id`s, and create a pandas DataFrame for each.

In [ ]:
# Collect all record set @id's
record_sets = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}.")

# For demo purposes, show columns for the first record set (if any)
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    print(f"Columns in first record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing fields, and grouping by key attributes.

We will pick the main survey data record set and work with typical numeric fields such as the PHQ-9 or GAD-7 total score, if available.

In [ ]:
# Identify main record set and a numeric field for demo (edit @ids if needed per the printed list above)
if len(record_sets) == 0:
    raise ValueError('No record sets found.')

# Replace these example @ids by those found above as appropriate
record_set_id = record_sets[0]
df = dataframes[record_set_id]
print(f"Working with record set: {record_set_id}\nColumns: {df.columns.tolist()}")

# Guess a numeric field by common survey score naming
numeric_field_candidates = [c for c in df.columns if any(x in c.lower() for x in ['phq9', 'gad7', 'psq', 'score'])]
if not numeric_field_candidates:
    # Fallback to pick the first float/integer-looking column
    if len(df.columns) > 0:
        numeric_field = df.columns[0]  # Adapt as needed
    else:
        raise Exception('No columns found in record set DataFrame.')
else:
    numeric_field = numeric_field_candidates[0]
print(f"Using numeric field for illustration: {numeric_field}")

# Set threshold (use actual data distribution!)
try:
    threshold = float(df[numeric_field].mean()) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
except Exception:
    threshold = 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
# Group (if present)
possible_group_fields = [c for c in df.columns if any(x in c.lower() for x in ["sex", "gender", "education", "village", "group"])]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print('No obvious group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields.

We'll plot a histogram for the selected numeric field and (optionally) boxplots grouped by gender (if the column exists).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
plt.hist(df[numeric_field].dropna(), bins=15, color='skyblue', edgecolor='black')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group field (e.g., gender/sex)
if possible_group_fields:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=possible_group_fields[0], y=numeric_field)
    plt.title(f"{numeric_field} by {possible_group_fields[0]}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides survey-based mental health indicators for Kilifi County, Kenya, including demographic and clinical score fields.
- Data was extracted using the Croissant schema and explored in both tabular and visual formats.
- Numeric analysis and grouping by demographic variables enable further statistical or ML modeling.

Continue with domain-specific analyses or machine learning using the processed data.